# 🏗️ Walls Structural Crack Detection — Deep Learning + Physics Analysis

Notebook ini mengklasifikasikan retak pada dinding beton menggunakan ResNet kustom, dengan analisis fisika material yang lengkap.

**Dataset**: `Walls.zip` — hanya subset Walls (1/8 dari total data)  
**Model**: ResNet dengan residual blocks  
**Analisis Fisika**: Morfologi retak, tekstur permukaan, intensitas, gradien, frekuensi spasial

---

## 0. Setup & Instalasi Dependensi

In [ ]:
# Install dependensi tambahan jika belum ada
!pip install -q scikit-image scipy wandb

## 1. Import Library

In [ ]:
import os
import random
import zipfile
import shutil
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader, Subset, random_split, ConcatDataset
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
from itertools import chain
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, classification_report,
    roc_curve, auc
)

# Physics / image analysis
from skimage import feature, filters, morphology, measure, exposure
from skimage.filters import sobel, laplace, gabor
from scipy import ndimage
from scipy.stats import entropy as scipy_entropy
import cv2

# GPU setup
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

## 2. Ekstraksi Dataset Walls dari ZIP

Dataset diekstrak dari `Walls.zip`. Struktur folder yang diharapkan:
```
Walls/
  Cracked/      ← label 0 (retak)
  Non-cracked/  ← label 1 (tidak retak)
```

In [ ]:
import os
import zipfile
import shutil

# ============================================================
# KONFIGURASI PATH — sudah diperbaiki
# ============================================================
# Colab: Walls.zip ada di root /
# Kaggle: biasanya di /kaggle/input/...
def find_zip(filename='Walls.zip'):
    search_dirs = ['/', '/content', '/kaggle/input', '.', '/root']
    for d in search_dirs:
        candidate = os.path.join(d, filename)
        if os.path.isfile(candidate):
            print(f"✅ Ditemukan: {candidate}")
            return candidate
    raise FileNotFoundError(
        f"{filename} tidak ditemukan. "
        f"Pastikan sudah diupload ke Colab (klik ikon folder lalu upload)."
    )

ZIP_PATH   = find_zip('Walls.zip')
EXTRACT_TO = './dataset_walls'

# ============================================================
# EKSTRAKSI
# ============================================================
if not os.path.exists(EXTRACT_TO):
    print("Mengekstrak Walls.zip ...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(EXTRACT_TO)
    print("Ekstraksi selesai!")
else:
    print("Dataset sudah diekstrak sebelumnya.")

# ============================================================
# FIX: Rename 'Non-cracked' → 'NonCracked'
# (tanda '-' kadang bermasalah di ImageFolder)
# ============================================================
for root_dir, dirs, _ in os.walk(EXTRACT_TO):
    for d in dirs:
        if d == 'Non-cracked':
            old_path = os.path.join(root_dir, d)
            new_path = os.path.join(root_dir, 'NonCracked')
            os.rename(old_path, new_path)
            print(f"Renamed: {old_path} → {new_path}")

# Deteksi struktur folder otomatis
walls_root = None
for root_dir, dirs, files in os.walk(EXTRACT_TO):
    subdirs_lower = [d.lower().replace('-', '') for d in dirs]
    if 'cracked' in subdirs_lower or 'noncracked' in subdirs_lower:
        walls_root = root_dir
        break

if walls_root is None:
    raise RuntimeError("Tidak menemukan folder Cracked/NonCracked. Periksa struktur ZIP.")

print(f"\nWalls root: {walls_root}")
for sub in os.listdir(walls_root):
    full = os.path.join(walls_root, sub)
    if os.path.isdir(full):
        imgs = [f for f in os.listdir(full) if f.lower().endswith(('.jpg','.png','.jpeg'))]
        print(f"  {sub}: {len(imgs)} gambar")

## 3. Konfigurasi Hyperparameter

In [ ]:
config = dict(
    epochs        = 10,
    num_classes   = 2,
    batch_size    = 32,
    learning_rate = 0.001,
    weight_decay  = 1e-4,
    image_size    = 128,      # diperkecil agar hemat memori
    data_slice    = 8,        # 1/8 dari total data
    train_ratio   = 0.8,
)
print("Konfigurasi:")
for k, v in config.items():
    print(f"  {k:20s}: {v}")

## 4. Dataset & DataLoader

Menggunakan `ImageFolder` dengan subset 1/8. Augmentasi data dilakukan khusus pada kelas **Cracked** yang lebih sedikit (untuk mengatasi ketidakseimbangan kelas).

In [ ]:
# ============================================================
# TRANSFORM
# ============================================================
IMG_SIZE = config['image_size']

base_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

# Augmentasi untuk kelas minority (Cracked)
aug_transform_flip = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Grayscale(num_output_channels=1),
    transforms.RandomHorizontalFlip(p=1.0),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

aug_transform_rotate = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Grayscale(num_output_channels=1),
    transforms.RandomRotation(degrees=45),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

aug_transform_blur = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Grayscale(num_output_channels=1),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

aug_transform_color = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Grayscale(num_output_channels=1),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])


# ============================================================
# LOAD DATASET — FULL (untuk eksplorasi)
# ============================================================
full_dataset = datasets.ImageFolder(root=walls_root, transform=base_transform)
print(f"Total gambar (full): {len(full_dataset)}")
print(f"Kelas: {full_dataset.classes}")
print(f"Mapping kelas: {full_dataset.class_to_idx}")

# Slice 1/8
SLICE = config['data_slice']
slice_indices = list(range(0, len(full_dataset), SLICE))
sliced_dataset = Subset(full_dataset, slice_indices)
print(f"\nTotal gambar (1/{SLICE} slice): {len(sliced_dataset)}")

# Hitung distribusi kelas setelah slice
slice_labels = [full_dataset.targets[i] for i in slice_indices]
for idx, cls in enumerate(full_dataset.classes):
    print(f"  {cls}: {slice_labels.count(idx)} gambar")

In [ ]:
# ============================================================
# TRAIN / TEST SPLIT + AUGMENTASI KELAS MINORITY
# ============================================================
n_total   = len(sliced_dataset)
n_train   = int(config['train_ratio'] * n_total)
n_test    = n_total - n_train

train_subset, test_subset = random_split(
    sliced_dataset, [n_train, n_test],
    generator=torch.Generator().manual_seed(SEED)
)

# Kelas mana yang 'Cracked'? biasanya idx=0 dari ImageFolder
cracked_idx = full_dataset.class_to_idx.get('Cracked', 0)
print(f"Index kelas Cracked: {cracked_idx}")

# Cari indeks cracked di train_subset
cracked_indices_in_train = [
    i for i in train_subset.indices
    if full_dataset.targets[i] == cracked_idx
]
print(f"Cracked di train sebelum aug: {len(cracked_indices_in_train)}")
non_cracked_count = n_train - len(cracked_indices_in_train)
print(f"Non-cracked di train: {non_cracked_count}")

# Buat augmented subsets
def make_aug_subset(base_dataset, indices, transform):
    """Subset dengan transform override."""
    sub = Subset(base_dataset, indices)
    # Wrap dengan custom dataset agar bisa override transform
    class AugDataset(torch.utils.data.Dataset):
        def __init__(self, subset, tfm):
            self.subset = subset
            self.tfm = tfm
        def __len__(self):
            return len(self.subset)
        def __getitem__(self, i):
            img_tensor, label = self.subset[i]
            # Denormalize → PIL → re-transform
            # Lebih mudah: load langsung dari path
            global_idx = self.subset.indices[i] if hasattr(self.subset, 'indices') else i
            path, lbl = full_dataset.samples[global_idx]
            from PIL import Image as PILImage
            img = PILImage.open(path).convert('RGB')
            return self.tfm(img), lbl
    return AugDataset(sub, transform)

# Augmented cracked datasets (4x augmentasi)
aug1 = make_aug_subset(full_dataset, cracked_indices_in_train, aug_transform_flip)
aug2 = make_aug_subset(full_dataset, cracked_indices_in_train, aug_transform_rotate)
aug3 = make_aug_subset(full_dataset, cracked_indices_in_train, aug_transform_blur)
aug4 = make_aug_subset(full_dataset, cracked_indices_in_train, aug_transform_color)

combined_train = ConcatDataset([train_subset, aug1, aug2, aug3, aug4])
print(f"\nTotal train setelah augmentasi: {len(combined_train)}")
print(f"Total test: {len(test_subset)}")

# DataLoaders
train_loader = DataLoader(
    combined_train, batch_size=config['batch_size'],
    shuffle=True, num_workers=2, pin_memory=True
)
test_loader = DataLoader(
    test_subset, batch_size=config['batch_size'],
    shuffle=False, num_workers=2, pin_memory=True
)

## 5. Eksplorasi Dataset — Visualisasi Sampel

In [ ]:
# Histogram distribusi kelas
class_names = full_dataset.classes
counts_full = [full_dataset.targets.count(i) for i in range(len(class_names))]
counts_slice = [slice_labels.count(i) for i in range(len(class_names))]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

colors = ['#e74c3c', '#3498db']
axes[0].bar(class_names, counts_full, color=colors, alpha=0.8, edgecolor='black')
axes[0].set_title('Distribusi Kelas — Dataset Penuh', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Jumlah Gambar')
for i, v in enumerate(counts_full):
    axes[0].text(i, v + 50, str(v), ha='center', fontweight='bold')

axes[1].bar(class_names, counts_slice, color=colors, alpha=0.8, edgecolor='black')
axes[1].set_title(f'Distribusi Kelas — Subset 1/{SLICE}', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Jumlah Gambar')
for i, v in enumerate(counts_slice):
    axes[1].text(i, v + 5, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('distribusi_kelas.png', dpi=150)
plt.show()
print(f"Rasio imbalance (Non-cracked/Cracked): {counts_full[1]/counts_full[0]:.2f}x")

In [ ]:
# Tampilkan 4 sampel per kelas
from PIL import Image as PILImage

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
fig.suptitle('Sampel Gambar per Kelas', fontsize=14, fontweight='bold')

for row, cls_name in enumerate(class_names):
    cls_idx = full_dataset.class_to_idx[cls_name]
    cls_paths = [p for p, l in full_dataset.samples if l == cls_idx]
    sampled = random.sample(cls_paths, 4)
    for col, path in enumerate(sampled):
        img = PILImage.open(path).convert('RGB')
        axes[row, col].imshow(img, cmap='gray')
        axes[row, col].set_title(cls_name, fontsize=10,
                                  color='red' if 'Crack' in cls_name else 'blue')
        axes[row, col].axis('off')

plt.tight_layout()
plt.savefig('sampel_gambar.png', dpi=150)
plt.show()

## 6. Analisis Fisika — Properti Material Dinding

Analisis fisika lengkap pada citra struktural mencakup:

| Aspek Fisika | Parameter | Relevansi |
|---|---|---|
| **Intensitas & Distribusi Cahaya** | Mean, Std, Histogram | Kontras permukaan |
| **Gradien & Tegangan Permukaan** | Sobel, Laplace, Edge density | Lokasi retak |
| **Morfologi Retak** | Skeletonisasi, panjang retak | Tipe kerusakan |
| **Tekstur Material** | GLCM (Haralick), Entropy | Kualitas beton |
| **Analisis Frekuensi Spasial** | FFT Power Spectrum | Pola periodik retak |
| **Kerapatan Energi Permukaan** | Energy density | Potensi propagasi |
| **Deteksi Tepi (Canny)** | Edge pixel ratio | Intensitas kerusakan |

In [ ]:
def compute_physics_features(img_path):
    """
    Ekstrak fitur fisika lengkap dari satu gambar dinding.

    Returns dict berisi semua fitur fisika:
    - Intensitas / Distribusi cahaya
    - Gradien (Sobel, Laplace) → tegangan permukaan
    - Morfologi retak (Canny, skeletonisasi)
    - Tekstur material (Haralick/GLCM, Entropy)
    - Frekuensi spasial (FFT power spectrum)
    - Kerapatan energi
    """
    # Load sebagai grayscale
    img_bgr = cv2.imread(img_path)
    if img_bgr is None:
        img_pil = PILImage.open(img_path).convert('L')
        img_gray = np.array(img_pil)
    else:
        img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    # Resize ke ukuran standar
    img_gray = cv2.resize(img_gray, (IMG_SIZE, IMG_SIZE))
    img_norm = img_gray.astype(np.float64) / 255.0

    features = {}

    # =========================================================
    # [1] INTENSITAS & DISTRIBUSI CAHAYA
    # Mengukur distribusi foton/cahaya yang dipantulkan permukaan.
    # Retak gelap → mean rendah, std tinggi
    # =========================================================
    features['intensity_mean']     = float(np.mean(img_norm))
    features['intensity_std']      = float(np.std(img_norm))
    features['intensity_min']      = float(np.min(img_norm))
    features['intensity_max']      = float(np.max(img_norm))
    features['intensity_range']    = features['intensity_max'] - features['intensity_min']
    # Skewness dan kurtosis distribusi piksel
    flat = img_norm.flatten()
    features['intensity_skewness'] = float(np.mean(((flat - flat.mean()) / (flat.std() + 1e-8))**3))
    features['intensity_kurtosis'] = float(np.mean(((flat - flat.mean()) / (flat.std() + 1e-8))**4) - 3)

    # =========================================================
    # [2] GRADIEN & TEGANGAN PERMUKAAN
    # Gradien intensitas ∝ perubahan tegangan mekanis di permukaan.
    # Retak memiliki gradien lokal yang sangat tinggi.
    # =========================================================
    # Sobel — gradien horizontal & vertikal
    sobel_x  = filters.sobel_h(img_norm)
    sobel_y  = filters.sobel_v(img_norm)
    grad_mag = np.sqrt(sobel_x**2 + sobel_y**2)
    features['gradient_mean']      = float(np.mean(grad_mag))
    features['gradient_std']       = float(np.std(grad_mag))
    features['gradient_max']       = float(np.max(grad_mag))
    # Arah gradien dominan (fisika: arah propagasi retak)
    grad_angle = np.arctan2(sobel_y, sobel_x + 1e-8)
    features['gradient_angle_std'] = float(np.std(grad_angle))

    # Laplacian — second-order, mendeteksi konsentrasi tegangan
    lap = laplace(img_norm)
    features['laplacian_energy']   = float(np.sum(lap**2))
    features['laplacian_std']      = float(np.std(lap))

    # =========================================================
    # [3] DETEKSI TEPI (EDGE) — CANNY
    # Tepi Canny merepresentasikan diskontinuitas permukaan.
    # Rasio piksel tepi tinggi → banyak retak
    # =========================================================
    img_uint8 = (img_norm * 255).astype(np.uint8)
    edges = cv2.Canny(img_uint8, threshold1=50, threshold2=150)
    features['edge_density']       = float(np.sum(edges > 0) / edges.size)
    features['edge_mean_intensity']= float(np.mean(edges))
    # Konektivitas tepi (retak linear memiliki tepi terhubung)
    labeled_edges, n_edge_comp = ndimage.label(edges > 0)
    features['edge_n_components']  = int(n_edge_comp)

    # =========================================================
    # [4] MORFOLOGI RETAK
    # Analisis geometri retak: panjang, lebar, orientasi.
    # Penting untuk menilai severity kerusakan struktural.
    # =========================================================
    # Threshold adaptif untuk isolasi retak
    thresh = filters.threshold_otsu(img_norm)
    binary_crack = img_norm < thresh  # piksel gelap = retak

    # Skeletonisasi — ekstrak garis tengah retak
    skeleton = morphology.skeletonize(binary_crack)
    features['crack_skeleton_length'] = int(np.sum(skeleton))

    # Area retak (pixel)
    crack_area = np.sum(binary_crack)
    features['crack_area_ratio']   = float(crack_area / binary_crack.size)

    # Properti region terkoneksi (komponen retak)
    labeled_crack = measure.label(binary_crack)
    props = measure.regionprops(labeled_crack)
    if props:
        areas = [p.area for p in props]
        features['crack_n_regions']    = len(props)
        features['crack_max_area']     = float(max(areas))
        features['crack_mean_area']    = float(np.mean(areas))
        # Eksentrisitas region terbesar (0=lingkaran, 1=garis lurus)
        largest = max(props, key=lambda p: p.area)
        features['crack_eccentricity'] = float(largest.eccentricity)
        # Orientasi retak dominan (dalam radian)
        features['crack_orientation']  = float(largest.orientation)
        features['crack_solidity']     = float(largest.solidity)
    else:
        features['crack_n_regions']    = 0
        features['crack_max_area']     = 0.0
        features['crack_mean_area']    = 0.0
        features['crack_eccentricity'] = 0.0
        features['crack_orientation']  = 0.0
        features['crack_solidity']     = 0.0

    # =========================================================
    # [5] TEKSTUR MATERIAL — GLCM (Haralick Features)
    # Gray-Level Co-occurrence Matrix mengukur pola tekstur
    # permukaan beton. Retak mengubah tekstur secara drastis.
    # =========================================================
    img_8bit = (img_norm * 255).astype(np.uint8)
    # Kuantisasi ke 8 level untuk efisiensi
    img_q    = (img_8bit // 32).astype(np.uint8)  # 0..7
    glcm = feature.graycomatrix(
        img_q, distances=[1], angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
        levels=8, symmetric=True, normed=True
    )
    features['glcm_contrast']    = float(feature.graycoprops(glcm, 'contrast').mean())
    features['glcm_dissimilarity']= float(feature.graycoprops(glcm, 'dissimilarity').mean())
    features['glcm_homogeneity'] = float(feature.graycoprops(glcm, 'homogeneity').mean())
    features['glcm_energy']      = float(feature.graycoprops(glcm, 'energy').mean())
    features['glcm_correlation'] = float(feature.graycoprops(glcm, 'correlation').mean())
    features['glcm_asm']         = float(feature.graycoprops(glcm, 'ASM').mean())

    # Entropi tekstur (Shannon entropy) — ketidakaturan permukaan
    hist, _ = np.histogram(img_norm, bins=64, range=(0,1), density=True)
    hist_nonzero = hist[hist > 0]
    features['texture_entropy']  = float(-np.sum(hist_nonzero * np.log2(hist_nonzero + 1e-8)))

    # =========================================================
    # [6] FREKUENSI SPASIAL — FFT POWER SPECTRUM
    # Analisis Fourier mengungkap pola periodik retak.
    # Retak memiliki konsentrasi energi di frekuensi tinggi.
    # =========================================================
    fft_img  = np.fft.fft2(img_norm)
    fft_shift= np.fft.fftshift(fft_img)
    power_spectrum = np.abs(fft_shift)**2
    # Normalisasi
    ps_norm  = power_spectrum / (power_spectrum.sum() + 1e-8)

    # Bagi ke low / mid / high frequency bands
    h, w = ps_norm.shape
    cy, cx = h // 2, w // 2
    Y, X   = np.ogrid[:h, :w]
    dist   = np.sqrt((X - cx)**2 + (Y - cy)**2)

    r_max  = min(h, w) // 2
    low    = ps_norm[dist < r_max * 0.2].sum()
    mid    = ps_norm[(dist >= r_max * 0.2) & (dist < r_max * 0.6)].sum()
    high   = ps_norm[dist >= r_max * 0.6].sum()

    features['fft_low_freq_energy']  = float(low)
    features['fft_mid_freq_energy']  = float(mid)
    features['fft_high_freq_energy'] = float(high)
    features['fft_high_low_ratio']   = float(high / (low + 1e-8))

    # =========================================================
    # [7] KERAPATAN ENERGI PERMUKAAN
    # Proporional terhadap energi regangan permukaan material.
    # E_surface ∝ ∫|∇I|² dA — tinggi pada area retak
    # =========================================================
    energy_density = np.mean(grad_mag**2)
    features['surface_energy_density'] = float(energy_density)

    # Variasi lokal energi (distribusi kerusakan)
    # Bagi gambar 4x4 block
    block_h, block_w = IMG_SIZE // 4, IMG_SIZE // 4
    block_energies = []
    for r in range(0, IMG_SIZE, block_h):
        for c in range(0, IMG_SIZE, block_w):
            block = grad_mag[r:r+block_h, c:c+block_w]
            block_energies.append(np.mean(block**2))
    features['energy_spatial_std']  = float(np.std(block_energies))
    features['energy_spatial_max']  = float(np.max(block_energies))

    return features

print("Fungsi compute_physics_features siap.")

In [ ]:
# Hitung fitur fisika untuk subset sampel (50 per kelas)
N_SAMPLES_PER_CLASS = 50

# Gunakan class_names langsung dari dataset (tidak hardcode)
# Setelah rename folder: class_names = ['Cracked', 'NonCracked']
physics_results = {cls: [] for cls in class_names}
print(f"Kelas yang terdeteksi: {class_names}")
print("Menghitung fitur fisika...")

for cls_name in class_names:
    cls_idx = full_dataset.class_to_idx[cls_name]

    # Ambil path dari slice indices saja
    cls_paths_slice = [
        full_dataset.samples[i][0]
        for i in slice_indices
        if full_dataset.targets[i] == cls_idx
    ]

    sampled = random.sample(
        cls_paths_slice,
        min(N_SAMPLES_PER_CLASS, len(cls_paths_slice))
    )

    for path in tqdm(sampled, desc=cls_name):
        feat = compute_physics_features(path)
        physics_results[cls_name].append(feat)

# Nama kelas untuk referensi di bawah
cracked_key     = [c for c in class_names if 'crack' in c.lower() and 'non' not in c.lower()][0]
noncracked_key  = [c for c in class_names if 'non' in c.lower() or 'noncracked' in c.lower()][0]

print(f"\nSelesai!")
print(f"  {cracked_key}    : {len(physics_results[cracked_key])} sampel")
print(f"  {noncracked_key} : {len(physics_results[noncracked_key])} sampel")

In [ ]:
# ============================================================
# VISUALISASI FISIKA: Perbandingan distribusi fitur per kelas
# ============================================================
import pandas as pd

df_cracked     = pd.DataFrame(physics_results[cracked_key])
df_non_cracked = pd.DataFrame(physics_results[noncracked_key])

# Pilih fitur terpenting untuk divisualisasikan
key_features = [
    ('intensity_mean',          'Intensitas Rata-rata',          'Distribusi cahaya\n(retak lebih gelap)'),
    ('gradient_mean',           'Gradien Rata-rata (Sobel)',     'Tegangan permukaan\n(retak memiliki gradien tinggi)'),
    ('edge_density',            'Kerapatan Tepi (Canny)',        'Densitas diskontinuitas\npermukaan'),
    ('crack_area_ratio',        'Rasio Area Retak',              'Proporsi permukaan retak\n(area gelap / total)'),
    ('crack_skeleton_length',   'Panjang Skeleton Retak',        'Panjang jalur retak\n(skeletonisasi morfologi)'),
    ('glcm_contrast',           'Kontras GLCM',                  'Ketidakseragaman tekstur\nmaterial beton'),
    ('texture_entropy',         'Entropi Tekstur',               'Ketidakaturan permukaan\n(Shannon entropy)'),
    ('fft_high_freq_energy',    'Energi Frekuensi Tinggi (FFT)', 'Komponen frekuensi tinggi\n(pola retak halus)'),
    ('surface_energy_density',  'Kerapatan Energi Permukaan',    'Energi regangan permukaan\n∝ ∫|∇I|² dA'),
    ('laplacian_energy',        'Energi Laplacian',              'Konsentrasi tegangan\n(second-order gradient)'),
]

fig, axes = plt.subplots(2, 5, figsize=(22, 9))
axes = axes.flatten()
fig.suptitle('Analisis Fisika: Perbandingan Cracked vs Non-cracked (Walls)',
             fontsize=15, fontweight='bold', y=1.01)

for ax, (feat, label, desc) in zip(axes, key_features):
    c_vals  = df_cracked[feat].dropna()
    nc_vals = df_non_cracked[feat].dropna()

    ax.hist(c_vals,  bins=20, alpha=0.7, color='#e74c3c', label='Cracked',     density=True)
    ax.hist(nc_vals, bins=20, alpha=0.7, color='#3498db', label='Non-cracked', density=True)
    ax.axvline(c_vals.mean(),  color='#c0392b', linestyle='--', linewidth=1.5)
    ax.axvline(nc_vals.mean(), color='#2980b9', linestyle='--', linewidth=1.5)
    ax.set_title(label, fontsize=9, fontweight='bold')
    ax.set_xlabel(desc, fontsize=7, color='gray')
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('analisis_fisika_distribusi.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# VISUALISASI FISIKA: 1 gambar retak dengan overlay
# ============================================================
cracked_paths = [p for p, l in full_dataset.samples
                 if l == full_dataset.class_to_idx.get('Cracked', 0)]
sample_path = random.choice(cracked_paths)

img_bgr  = cv2.imread(sample_path)
img_rgb  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
img_gray_sample = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
img_gray_sample = cv2.resize(img_gray_sample, (IMG_SIZE, IMG_SIZE))
img_norm_sample = img_gray_sample.astype(np.float64) / 255.0

# Hitung turunan
sobel_x  = filters.sobel_h(img_norm_sample)
sobel_y  = filters.sobel_v(img_norm_sample)
grad_mag = np.sqrt(sobel_x**2 + sobel_y**2)
lap      = laplace(img_norm_sample)
edges    = cv2.Canny(img_gray_sample, 50, 150)
thresh   = filters.threshold_otsu(img_norm_sample)
binary   = img_norm_sample < thresh
skel     = morphology.skeletonize(binary)

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
fig.suptitle('Analisis Fisika — Satu Gambar Retak (Walls)',
             fontsize=13, fontweight='bold')

panels = [
    (cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE)), 'Original (RGB)', None),
    (img_norm_sample,    'Grayscale Ternormalisasi\n(Distribusi Intensitas)', 'gray'),
    (grad_mag,           'Gradien Sobel\n(Tegangan Permukaan)', 'hot'),
    (np.abs(lap),        'Laplacian (Abs)\n(Konsentrasi Tegangan)', 'plasma'),
    (edges,              'Tepi Canny\n(Diskontinuitas Permukaan)', 'gray'),
    (binary.astype(float), 'Otsu Threshold\n(Isolasi Area Retak)', 'gray'),
    (skel.astype(float), 'Skeleton Retak\n(Geometri & Panjang Jalur)', 'gray'),
    (np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(img_norm_sample)))),
     'Log FFT Power Spectrum\n(Frekuensi Spasial)', 'viridis'),
]

for ax, (data, title, cmap) in zip(axes.flatten(), panels):
    if cmap:
        ax.imshow(data, cmap=cmap)
    else:
        ax.imshow(data)
    ax.set_title(title, fontsize=9, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.savefig('analisis_fisika_satu_gambar.png', dpi=150)
plt.show()

In [ ]:
# Tabel ringkasan statistik fitur fisika
summary_features = [
    'intensity_mean', 'intensity_std', 'gradient_mean', 'gradient_max',
    'laplacian_energy', 'edge_density', 'crack_area_ratio',
    'crack_skeleton_length', 'crack_eccentricity', 'glcm_contrast',
    'glcm_homogeneity', 'texture_entropy', 'fft_high_freq_energy',
    'surface_energy_density'
]

print(f"{'Fitur Fisika':<28} {'Cracked (mean±std)':>22} {'Non-cracked (mean±std)':>24} {'Δ (%)':>10}")
print("-" * 90)
for f in summary_features:
    c_mean  = df_cracked[f].mean()
    c_std   = df_cracked[f].std()
    nc_mean = df_non_cracked[f].mean()
    nc_std  = df_non_cracked[f].std()
    delta   = (c_mean - nc_mean) / (abs(nc_mean) + 1e-8) * 100
    print(f"{f:<28} {c_mean:>10.4f}±{c_std:<10.4f} {nc_mean:>10.4f}±{nc_std:<10.4f} {delta:>+9.1f}%")

## 7. Arsitektur Model: ResNet Kustom (Ditingkatkan)

Arsitektur ditingkatkan dari baseline:
- **Dropout** pada fully-connected layer untuk regularisasi
- **4 ResNet layers** (vs 3 di baseline) untuk representasi lebih dalam
- **Batch Normalization** pada setiap blok
- **Input: grayscale (1 channel)**, sesuai analisis fisika

In [ ]:
class ResidualBlock(nn.Module):
    """
    Blok residual standar (BasicBlock) dengan skip connection.
    Terinspirasi dari He et al. (2016) — Deep Residual Learning.

    Skip connection menyelesaikan masalah vanishing gradient
    pada jaringan dalam, memungkinkan training yang lebih stabil.
    """
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1, downsample=None, dropout_rate=0.1):
        super().__init__()
        self.conv1   = nn.Conv2d(in_channels, out_channels, kernel_size=3,
                                  stride=stride, padding=1, bias=False)
        self.bn1     = nn.BatchNorm2d(out_channels)
        self.relu    = nn.ReLU(inplace=True)
        self.conv2   = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                                  stride=1, padding=1, bias=False)
        self.bn2     = nn.BatchNorm2d(out_channels)
        self.downsample = downsample
        self.dropout = nn.Dropout2d(p=dropout_rate)

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.dropout(out)

        out = self.conv2(out)
        out = self.bn2(out)

        if self.downsample:
            identity = self.downsample(x)

        out = out + identity   # residual connection
        out = self.relu(out)
        return out


class WallsResNet(nn.Module):
    """
    ResNet kustom untuk deteksi retak pada dinding beton.

    Modifikasi dari baseline:
    - 4 layer (bukan 3) untuk fitur hierarkis lebih kaya
    - Dropout pada classifier
    - Input grayscale (1 channel)
    - AdaptiveAvgPool untuk fleksibilitas ukuran input
    """
    def __init__(self, num_classes=2, dropout_rate=0.3):
        super().__init__()
        self.in_channels = 64

        # Stem: ekstraksi fitur awal
        self.stem = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        )

        # ResNet layers
        self.layer1 = self._make_layer(64,  64,  n_blocks=2, stride=1)
        self.layer2 = self._make_layer(64,  128, n_blocks=2, stride=2)
        self.layer3 = self._make_layer(128, 256, n_blocks=2, stride=2)
        self.layer4 = self._make_layer(256, 512, n_blocks=2, stride=2)

        # Classifier
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout_rate),
            nn.Linear(512, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout_rate / 2),
            nn.Linear(128, num_classes)
        )

        # Weight initialization
        self._initialize_weights()

    def _make_layer(self, in_ch, out_ch, n_blocks, stride):
        downsample = None
        if stride != 1 or in_ch != out_ch:
            downsample = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch)
            )
        layers = [ResidualBlock(in_ch, out_ch, stride=stride, downsample=downsample)]
        self.in_channels = out_ch
        for _ in range(1, n_blocks):
            layers.append(ResidualBlock(out_ch, out_ch))
        return nn.Sequential(*layers)

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x


# Inisiasi model
model = WallsResNet(num_classes=config['num_classes']).to(device)

# Hitung parameter
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameter    : {total_params:,}")
print(f"Parameter trainable: {trainable_params:,}")
print(f"\nArsitektur:")
print(model)

## 8. Training

- **Loss**: CrossEntropyLoss dengan class weight (mengatasi imbalance)
- **Optimizer**: Adam dengan weight decay
- **Scheduler**: CosineAnnealingLR untuk learning rate decay halus

In [ ]:
# ============================================================
# LOSS FUNCTION — Weighted CrossEntropy
# Weight berbanding terbalik dengan frekuensi kelas
# ============================================================
n_cracked     = slice_labels.count(cracked_idx)
n_non_cracked = len(slice_labels) - n_cracked
total_samples = len(slice_labels)

# Weight = total / (n_classes * count_per_class)
w_cracked     = total_samples / (2 * n_cracked)
w_non_cracked = total_samples / (2 * n_non_cracked)

class_weights = torch.tensor([w_cracked, w_non_cracked], dtype=torch.float).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

print(f"Class weights — Cracked: {w_cracked:.4f}, Non-cracked: {w_non_cracked:.4f}")

# Optimizer
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=config['learning_rate'],
    weight_decay=config['weight_decay']
)

# Learning Rate Scheduler
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=config['epochs'], eta_min=1e-6
)

In [ ]:
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        # Gradient clipping untuk stabilitas
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total   += labels.size(0)

    return total_loss / total, correct / total


def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_labels, all_preds, all_probs = [], [], []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            probs = F.softmax(outputs, dim=1)

            total_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total   += labels.size(0)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(predicted.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())

    return (total_loss / total, correct / total,
            all_labels, all_preds, all_probs)


# ============================================================
# TRAINING LOOP
# ============================================================
history = {
    'train_loss': [], 'train_acc': [],
    'val_loss':   [], 'val_acc':   [], 'lr': []
}

best_val_acc = 0.0
best_model_state = None

print(f"Mulai training: {config['epochs']} epoch, batch size={config['batch_size']}")
print("-" * 70)

for epoch in range(config['epochs']):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc, _, _, _ = evaluate(model, test_loader, criterion)
    scheduler.step()

    current_lr = optimizer.param_groups[0]['lr']
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['lr'].append(current_lr)

    # Simpan model terbaik
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_state = {k: v.clone() for k, v in model.state_dict().items()}

    print(f"Epoch [{epoch+1:02d}/{config['epochs']}] "
          f"Train Loss: {train_loss:.4f}  Train Acc: {train_acc*100:.2f}%  "
          f"Val Loss: {val_loss:.4f}  Val Acc: {val_acc*100:.2f}%  "
          f"LR: {current_lr:.6f}")

print(f"\n✅ Training selesai! Best Val Acc: {best_val_acc*100:.2f}%")

## 9. Evaluasi Model

In [ ]:
# Load model terbaik
model.load_state_dict(best_model_state)
_, final_acc, all_labels, all_preds, all_probs = evaluate(model, test_loader, criterion)

print(f"\n{'='*55}")
print(f" LAPORAN EVALUASI AKHIR")
print(f"{'='*55}")
print(f" Akurasi       : {final_acc*100:.4f}%")
print(f"{'='*55}")
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names))

In [ ]:
# Training curves
epochs_range = range(1, config['epochs'] + 1)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(epochs_range, history['train_loss'], 'r-o', label='Train', markersize=4)
axes[0].plot(epochs_range, history['val_loss'],   'b-o', label='Val',   markersize=4)
axes[0].set_title('Loss per Epoch', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(alpha=0.4)

axes[1].plot(epochs_range, [a*100 for a in history['train_acc']], 'r-o', label='Train', markersize=4)
axes[1].plot(epochs_range, [a*100 for a in history['val_acc']],   'b-o', label='Val',   markersize=4)
axes[1].set_title('Akurasi per Epoch', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
axes[1].legend(); axes[1].grid(alpha=0.4)

axes[2].plot(epochs_range, history['lr'], 'g-o', markersize=4)
axes[2].set_title('Learning Rate (Cosine Annealing)', fontweight='bold')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('LR')
axes[2].grid(alpha=0.4)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

In [ ]:
# Confusion Matrix + ROC Curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix (dinormalisasi)
cm = confusion_matrix(all_labels, all_preds, normalize='true')
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(cmap='Blues', values_format='.3f', colorbar=False, ax=axes[0])
axes[0].set_title(f'Confusion Matrix (Normalized)\nAkurasi: {final_acc*100:.2f}%',
                   fontweight='bold')

# ROC Curve
fpr, tpr, _ = roc_curve(all_labels, all_probs)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, 'b-', lw=2, label=f'ROC Curve (AUC = {roc_auc:.4f})')
axes[1].plot([0,1], [0,1], 'k--', lw=1, label='Random Classifier')
axes[1].fill_between(fpr, tpr, alpha=0.1, color='blue')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve', fontweight='bold')
axes[1].legend(); axes[1].grid(alpha=0.4)

plt.tight_layout()
plt.savefig('confusion_roc.png', dpi=150)
plt.show()

print(f"AUC-ROC: {roc_auc:.4f}")

In [ ]:
# Visualisasi prediksi: benar & salah
model.eval()
result_images = {'TP': [], 'TN': [], 'FP': [], 'FN': []}

with torch.no_grad():
    for images, labels in test_loader:
        images_dev = images.to(device)
        outputs    = model(images_dev)
        _, preds   = outputs.max(1)
        preds_cpu  = preds.cpu()

        for i in range(len(labels)):
            true_l = labels[i].item()
            pred_l = preds_cpu[i].item()
            img_t  = images[i]

            if true_l == 0 and pred_l == 0 and len(result_images['TP']) < 3:
                result_images['TP'].append((img_t, true_l, pred_l))
            elif true_l == 1 and pred_l == 1 and len(result_images['TN']) < 3:
                result_images['TN'].append((img_t, true_l, pred_l))
            elif true_l == 0 and pred_l == 1 and len(result_images['FN']) < 3:
                result_images['FN'].append((img_t, true_l, pred_l))
            elif true_l == 1 and pred_l == 0 and len(result_images['FP']) < 3:
                result_images['FP'].append((img_t, true_l, pred_l))

        if all(len(v) >= 3 for v in result_images.values()):
            break

fig, axes = plt.subplots(4, 3, figsize=(12, 16))
titles_map = {
    'TP': ('✅ Benar: Cracked → Cracked',    'green'),
    'TN': ('✅ Benar: Non-cracked → Non-cracked', 'blue'),
    'FN': ('❌ Salah: Cracked → Non-cracked', 'red'),
    'FP': ('❌ Salah: Non-cracked → Cracked', 'orange'),
}

for row_idx, (key, (title, color)) in enumerate(titles_map.items()):
    imgs = result_images[key]
    for col_idx in range(3):
        ax = axes[row_idx, col_idx]
        if col_idx < len(imgs):
            img_show = imgs[col_idx][0].squeeze().numpy()
            ax.imshow(img_show, cmap='gray')
        ax.axis('off')
        if col_idx == 1:
            ax.set_title(title, fontsize=9, color=color, fontweight='bold')

plt.suptitle('Contoh Prediksi Model', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('contoh_prediksi.png', dpi=150)
plt.show()

## 10. Ringkasan Analisis Fisika Material

Berdasarkan hasil ekstrasi fitur fisika dan performa model, berikut interpretasi fisika dari hasil klasifikasi:

In [ ]:
# Ringkasan perbedaan fisika Cracked vs Non-cracked
print("="*70)
print(" RINGKASAN ANALISIS FISIKA MATERIAL DINDING")
print("="*70)

interpretations = [
    ("1. Distribusi Intensitas Cahaya",
     "Retak menyerap lebih banyak cahaya → intensitas rata-rata lebih RENDAH.",
     "Std deviasi lebih TINGGI karena kontras tajam antara retak dan beton.",
     'intensity_mean', 'intensity_std'),

    ("2. Gradien & Tegangan Permukaan",
     "Gradien Sobel tinggi di area retak mengindikasikan DISKONTINUITAS permukaan.",
     "Secara fisika: ∇I ∝ tegangan lokal permukaan → retak = zona tegangan tinggi.",
     'gradient_mean', 'laplacian_energy'),

    ("3. Morfologi Retak",
     "Skeleton retak lebih PANJANG pada Cracked → kerusakan struktural signifikan.",
     "Eksentrisitas tinggi → retak LINEAR (bukan melingkar) → crack propagation.",
     'crack_skeleton_length', 'crack_eccentricity'),

    ("4. Tekstur Material (GLCM)",
     "Kontras GLCM LEBIH TINGGI pada Cracked → ketidakseragaman tekstur beton.",
     "Homogenitas LEBIH RENDAH → distribusi piksel tidak seragam (ada retak).",
     'glcm_contrast', 'glcm_homogeneity'),

    ("5. Frekuensi Spasial (FFT)",
     "Energi frekuensi TINGGI lebih besar pada Cracked → tepi retak tajam.",
     "Non-cracked lebih dominan di frekuensi rendah → permukaan halus/homogen.",
     'fft_high_freq_energy', 'fft_high_low_ratio'),

    ("6. Kerapatan Energi Permukaan",
     "E_surface = ∫|∇I|²dA lebih TINGGI pada Cracked.",
     "Mengindikasikan zona akumulasi energi regangan sebelum kegagalan material.",
     'surface_energy_density', 'energy_spatial_std'),
]

for title, desc1, desc2, feat1, feat2 in interpretations:
    c1  = df_cracked[feat1].mean()
    nc1 = df_non_cracked[feat1].mean()
    c2  = df_cracked[feat2].mean()
    nc2 = df_non_cracked[feat2].mean()

    print(f"\n{title}")
    print(f"  {desc1}")
    print(f"  {desc2}")
    print(f"  {feat1:<30}: Cracked={c1:.5f}  vs  Non-cracked={nc1:.5f}")
    print(f"  {feat2:<30}: Cracked={c2:.5f}  vs  Non-cracked={nc2:.5f}")

print("\n" + "="*70)
print(f" Model Accuracy: {final_acc*100:.2f}%  |  AUC-ROC: {roc_auc:.4f}")
print("="*70)

## 11. Simpan Model

In [ ]:
# Simpan weights model
torch.save({
    'model_state_dict': best_model_state,
    'config': config,
    'best_val_acc': best_val_acc,
    'class_names': class_names,
    'history': history,
}, 'walls_resnet_best.pth')

print("Model disimpan ke: walls_resnet_best.pth")
print(f"Best validation accuracy: {best_val_acc*100:.4f}%")

# Export ONNX (opsional)
try:
    model.eval()
    dummy_input = torch.randn(1, 1, IMG_SIZE, IMG_SIZE).to(device)
    torch.onnx.export(
        model, dummy_input, 'walls_resnet.onnx',
        input_names=['image'], output_names=['class_logits'],
        opset_version=11
    )
    print("Model juga diekspor ke: walls_resnet.onnx")
except Exception as e:
    print(f"ONNX export gagal (opsional): {e}")